In [27]:
import pandas as pd
import numpy as np

import scipy.io as sio
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams['font.size'] = 8.0
matplotlib.rcParams['pdf.fonttype'] = 42

In [28]:
def getTrialAlignment(index, unittp):
    repeats = []
    for idx_st, idx_stop in zip(index['index_starts'], index['index_stops']):
        rep = unittp[(unittp > idx_st) & (unittp < idx_stop)].values
        rep = rep - idx_st
        repeats.append(rep)
    return repeats

In [29]:
for k in all_Units_aligned.keys():
    print(k, len(all_Units_aligned[k]), 'repeats')
    print('spikes in first 3 repeats:', [len(r) for r in all_Units_aligned[k][:3]])

unit_0 30 repeats
spikes in first 3 repeats: [261, 233, 255]
unit_1 30 repeats
spikes in first 3 repeats: [431, 442, 433]
unit_2 30 repeats
spikes in first 3 repeats: [139, 143, 150]


In [30]:
# input all parameters
Fs = 3e4
TimeHistory = 20e-3 # in seconds
TimePost = 2e-3 # in seconds

In [31]:
# read in the index based on stimulus repeats
name = 'm30_wn1'
index = pd.read_csv('/Users/meghnaanand/Desktop/Analysis/' + name + '_index.csv')
# convert index to timestamp
index_tp = index * 1/Fs

In [32]:
# read in the whiteNoise stimulus
data = pd.read_csv('/Users/meghnaanand/Desktop/Analysis/' + name + '_degrees.csv')
Stim = data['angle_degrees']

# get the first WN repeat 
start = index.loc[0, :][0]
stop = index.loc[0, :][1]
WhiteNoise = Stim[start:stop]
# reindex the WhiteNoise to reflect in seconds
WhiteNoise.index = np.arange(0, len(WhiteNoise))*1/Fs

/var/folders/__/mwz7g0652137ypy1vqmvdp3c0000gn/T/ipykernel_11693/1278819417.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  start = index.loc[0, :][0]
/var/folders/__/mwz7g0652137ypy1vqmvdp3c0000gn/T/ipykernel_11693/1278819417.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  stop = index.loc[0, :][1]


In [ ]:
print(WhiteNoise[:5])
print(WhiteNoise.min(), WhiteNoise.max())

In [33]:
# get spike timestamp
filename = '/Users/meghnaanand/Desktop/Analysis/m30_wn1_07.txt'
T = pd.read_csv(filename, sep = '\s+')
tp = T.loc[:,'Timestamp']

#loop through all units
NumUnits = np.unique(T['Unit'])
all_Units = {}
for idx, unit in enumerate(NumUnits):
    all_Units['unit_' + str(idx)] = T.loc[T['Unit'] == unit, 'Timestamp']
    
# align the spikes to stimulus repeats
all_Units_aligned = {}
for k in all_Units.keys():
    repeats = getTrialAlignment(index_tp, all_Units[k])
    all_Units_aligned[k] = repeats

In [34]:
# get prior timestamp
import random

num_repeats = len(index_tp)
num_permuts = 500
all_Units_prior = {}
for k in all_Units_aligned.keys():
    ## decide how many repeats to we want to compute for the prior 
    ## and also how many spikes in each repeat
#     l = []
#     for eachRep in all_Units_aligned[k]:
#         l.append(len(eachRep))
#     num_permuts = int(np.mean(l))  
    randomlist = [ [ None for y in range(num_permuts)]
             for x in range(num_repeats)]
    for jj in range(num_repeats):
        for ii in range(num_permuts):
            n = random.randint(int(20e-3 * Fs), len(WhiteNoise) - 1) / Fs
            randomlist[jj][ii] = n
    all_Units_prior[k] = randomlist

In [35]:
# pickle dump the timestamps
import pickle

saveObject = (all_Units_aligned,all_Units_prior, WhiteNoise)
with open('/Users/meghnaanand/Desktop/Analysis/' + name + '_timeStamps.pickle',"wb") as f:
    pickle.dump(saveObject, f)

In [9]:
print(T.head())
print(T.columns.tolist())
print(NumUnits)

   Timestamp  Unit
0   0.378933     0
1   1.047800     0
2   1.066533     0
3   1.081800     3
4   1.089767     0
['Timestamp', 'Unit']
[0 1 2 3]


In [10]:
print(index_tp.head())
print(index_tp.shape)

   index_starts  index_stops
0      1.042433    11.042533
1     13.106333    23.106400
2     25.170200    35.170300
3     37.234100    47.234167
4     49.297967    59.298067
(30, 2)


In [12]:
import pickle
with open('/Users/meghnaanand/Desktop/Analysis/m31_wn2_timeStamps.pickle', 'rb') as f:
    aligned, prior, WN = pickle.load(f)
print(aligned.keys())
print(len(WN))

dict_keys(['unit_0', 'unit_1', 'unit_2', 'unit_3'])
300003
